# Step 7: Ridge and Lasso (keeping the model calm)

In [5]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

X_train = pd.read_csv("../data/processed/X_train_clip.csv")
X_test = pd.read_csv("../data/processed/X_test_clip.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [6]:
# Same scoring helper as Step 6
def score_model(name, model, X_te, y_te):
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    print(f"{name}")
    print("  Accuracy :", round(accuracy_score(y_te, pred), 3))
    print("  Precision:", round(precision_score(y_te, pred, zero_division=0), 3))
    print("  Recall   :", round(recall_score(y_te, pred), 3))
    print("  F1       :", round(f1_score(y_te, pred), 3))
    print("  ROC-AUC  :", round(roc_auc_score(y_te, proba), 3))

In [7]:
# Ridge: try different brake strengths, pick the best using cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {"C": [0.001, 0.01, 0.1, 1, 10]}

ridge = GridSearchCV(
    LogisticRegression(penalty="l2", solver="liblinear", class_weight="balanced"),
    param_grid, cv=cv, scoring="roc_auc"
)
ridge.fit(X_train_s, y_train)

print("Best C for Ridge:", ridge.best_params_["C"])
print("Cross-validation ROC-AUC:", round(ridge.best_score_, 3))
score_model("Ridge", ridge.best_estimator_, X_test_s, y_test)

Best C for Ridge: 10
Cross-validation ROC-AUC: 0.742
Ridge
  Accuracy : 0.716
  Precision: 0.105
  Recall   : 0.667
  F1       : 0.181
  ROC-AUC  : 0.741


In [8]:
# Lasso: same idea, but weak features get pushed to zero
lasso = GridSearchCV(
    LogisticRegression(penalty="l1", solver="liblinear", class_weight="balanced"),
    param_grid, cv=cv, scoring="roc_auc"
)
lasso.fit(X_train_s, y_train)

print("Best C for Lasso:", lasso.best_params_["C"])
print("Cross-validation ROC-AUC:", round(lasso.best_score_, 3))
score_model("Lasso", lasso.best_estimator_, X_test_s, y_test)

Best C for Lasso: 1
Cross-validation ROC-AUC: 0.744
Lasso
  Accuracy : 0.712
  Precision: 0.102
  Recall   : 0.657
  F1       : 0.177
  ROC-AUC  : 0.741


In [9]:
# Which features did Lasso keep?
coefs = pd.Series(lasso.best_estimator_.coef_[0], index=X_train.columns)
kept = coefs[coefs != 0]
print("Features kept:", len(kept), "of", len(coefs))
print()
print(kept.abs().sort_values(ascending=False).head(10).round(3))

Features kept: 61 of 63

Attr63    2.397
Attr1     2.035
Attr33    2.017
Attr46    1.522
Attr13    1.354
Attr48    1.023
Attr18    0.971
Attr51    0.926
Attr54    0.856
Attr23    0.825
dtype: float64
